In [ ]:
from dotenv import load_dotenv

load_dotenv()

from langchain_groq import ChatGroq
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

documents = [

    Document(page_content="Python is a programming language."),

    Document(page_content="LangChain is an LLM framework."),

    Document(page_content="Machine Learning is a subset of AI."),

    Document(page_content="Deep Learning uses neural networks.")

]

vectorstore = FAISS.from_documents(
    documents,
    embeddings
)

# Vector Store Retriever

In [ ]:
retriever = vectorstore.as_retriever()

results = retriever.invoke(
    "What is LangChain?"
)

for doc in results:
    print(doc.page_content)

# Retriever with Top-K Results

In [ ]:
retriever = vectorstore.as_retriever(

    search_kwargs={
        "k":2
    }

)

results = retriever.invoke(
    "Artificial Intelligence"
)

for doc in results:
    print(doc.page_content)

# Similarity Search Retriever

In [ ]:
retriever = vectorstore.as_retriever(

    search_type="similarity",

    search_kwargs={
        "k":3
    }

)

results = retriever.invoke(
    "Python"
)

for doc in results:
    print(doc.page_content)

# MMR Retriever (Maximum Marginal Relevance)

In [ ]:
retriever = vectorstore.as_retriever(

    search_type="mmr",

    search_kwargs={
        "k":3,
        "fetch_k":5
    }

)

results = retriever.invoke(
    "Artificial Intelligence"
)

for doc in results:
    print(doc.page_content)

# Similarity Score Threshold Retriever

In [ ]:
retriever = vectorstore.as_retriever(

    search_type="similarity_score_threshold",

    search_kwargs={
        "score_threshold":0.7
    }

)

results = retriever.invoke(
    "Machine Learning"
)

for doc in results:
    print(doc.page_content)

# MultiQueryRetriever

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever

retriever = MultiQueryRetriever.from_llm(

    retriever=vectorstore.as_retriever(),

    llm=llm

)

results = retriever.invoke(
    "Explain Artificial Intelligence"
)

for doc in results:
    print(doc.page_content)

# Contextual Compression Retriever

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(

    base_compressor=compressor,

    base_retriever=vectorstore.as_retriever()

)

results = compression_retriever.invoke(
    "Explain LangChain"
)

for doc in results:
    print(doc.page_content)

# Parent Document Retriever

In [ ]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200
)

store = InMemoryStore()

retriever = ParentDocumentRetriever(

    vectorstore=vectorstore,

    docstore=store,

    child_splitter=child_splitter,

    parent_splitter=parent_splitter

)

# TimeWeightedVectorStoreRetriever

In [ ]:
from langchain.retrievers import TimeWeightedVectorStoreRetriever
from langchain.storage import InMemoryStore

retriever = TimeWeightedVectorStoreRetriever(

    vectorstore=vectorstore,

    memory_stream=[],

    decay_rate=0.01,

    k=2

)

# Retriever + Chat Model

In [ ]:
retriever = vectorstore.as_retriever()

documents = retriever.invoke(
    "What is Machine Learning?"
)

context = "\n".join(
    doc.page_content
    for doc in documents
)

response = llm.invoke(
    f"""
Context:
{context}

Question:
What is Machine Learning?
"""
)

print(response.content)

# Custom Retriever

In [ ]:
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document

class CustomRetriever(BaseRetriever):

    docs: list[Document]

    def _get_relevant_documents(self, query: str):

        return self.docs

documents = [

    Document(page_content="Python"),

    Document(page_content="LangChain")

]

retriever = CustomRetriever(
    docs=documents
)

results = retriever.invoke(
    "Anything"
)

for doc in results:
    print(doc.page_content)